# 03 — Preprocessing
Clean, encode, scale and split the data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib, os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(df.shape)

In [ ]:
# Keep only useful features
FEATURES = ['accommodates', 'bedrooms', 'bathrooms',
            'neighbourhood_cleansed', 'room_type',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews',
            'reviews_per_month']
TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')

In [ ]:
# Fill missing numerics
for col in ['bedrooms', 'bathrooms', 'reviews_per_month']:
    df[col] = df[col].fillna(df[col].median())

# One-hot encode categoricals
df = pd.get_dummies(df, columns=['neighbourhood_cleansed', 'room_type'], drop_first=True)
print(f'After encoding: {df.shape}')
df.head()

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
scaler = StandardScaler()
num_cols = ['accommodates', 'bedrooms', 'bathrooms',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews', 'reviews_per_month']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

# Save splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)
print('Splits saved.')